![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# 🏘️ Population Age & Gender — Data Cleaning Pipeline
![Python](https://img.shields.io/badge/Python_3.12-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![SQLite](https://img.shields.io/badge/SQLite-1A6B5A?style=flat-square&logo=sqlite&logoColor=white) ![Source](https://img.shields.io/badge/Source-Statistics_Canada-1B3A6B?style=flat-square)

> **Analytical Question:** How has Nova Scotia's population age distribution shifted over the past decade, and what are the projected trends?

This notebook cleans the raw Statistics Canada population estimates for Nova Scotia and produces a tidy, long-format dataset ready for Power BI and forecasting analysis.

---

## 📦 Source Data

| File | Gender | Age Groups | Years |
|------|--------|------------|-------|
| `Nova Scotia Men population Age & Gender(2012-2025).csv` | Male | 21 bands + All ages | 2012–2025 |
| `Nova Scotia Women Population Age & Gender(2012 - 2025).csv` | Female | 21 bands + All ages | 2012–2025 |

**Source:** Statistics Canada, Table 17-10-0005-01 — *Population estimates on July 1, by age and gender*

---

## 🔧 Cleaning Steps Overview

| Step | Action |
|------|--------|
| 1 | Imports & file paths |
| 2 | Smart StatCan reader — detects data boundaries, skips metadata |
| 3 | Remove non-data rows (unit labels, blanks) |
| 4 | Unpivot wide → long format (`pd.melt`) |
| 5 | Cast types, strip comma formatting |
| 6 | Combine Men + Women, add Gender label |
| 7 | Save clean CSV + load to SQLite |

---


## Step 1 — Imports & File Paths

We use `pandas` for data wrangling, `sqlite3` (built into Python) as our lightweight database, and `StringIO` to parse extracted text blocks without writing temporary files.


In [ ]:
import pandas as pd
import sqlite3
from io import StringIO

# ── File paths ──────────────────────────────────────────────
men_file   = "./data/Nova Scotia Men population Age & Gender(2012-2025).csv"
women_file = "./data/Nova Scotia Women Population Age & Gender(2012 - 2025).csv"

output_csv = "./clean/clean_population.csv"
sqlite_db  = "./database/population.db"


## Step 2 — Smart StatCan Table Reader


---

### Why we need a custom reader

StatCan CSVs are not clean flat files. They contain:
- **5–10 metadata rows** at the top (title, frequency, table number, release date)
- **Unit label rows** (`Persons`, `Years`) mixed into the data
- **Footnote rows** at the bottom (numbered citations)
- A **`Median age`** row that is a DECIMAL, not a population count

A standard `pd.read_csv()` would import all of this garbage.

### Strategy
We scan the file line-by-line, find the row containing `"Age group"` (the real header), and stop at `"Median age"` or `"Footnotes"`. Only the slice between those boundaries is parsed.

```
Row 1-8:  metadata + blanks    ← SKIP
Row 9-10: Geography / Gender   ← SKIP
Row 11:   "Age group | 2012 ... 2025"  ← START HERE
Row 12:   "Persons" unit label  ← skip
Row 13+:  actual data          ← KEEP
Row 35:   "Median age"         ← STOP
Row 36+:  footnotes            ← IGNORE
```


In [ ]:
def read_statscan_table(file_path):
    """
    Extracts the real data table from a StatCan CSV,
    skipping all metadata, unit labels, and footnotes.

    Strategy:
    - Scan lines to find the row with 'Age group' (real header)
    - Stop at 'Median age' or 'Footnotes' (end of data)
    - Parse only the extracted slice
    """
    with open(file_path, "r", encoding="utf-8-sig", errors="ignore") as f:
        lines = f.readlines()

    start_idx = None
    end_idx   = None

    for i, line in enumerate(lines):
        if start_idx is None and "Age group" in line:
            start_idx = i
        elif start_idx is not None and ("Median age" in line or "Footnotes" in line):
            end_idx = i
            break

    if start_idx is None:
        raise ValueError(f"Could not locate StatsCan table header in {file_path}")

    if end_idx is None:
        end_idx = len(lines)

    # Parse only the clean data slice
    table_text = "".join(lines[start_idx:end_idx])
    return pd.read_csv(StringIO(table_text))


## Step 3 — Clean & Reshape One File

Each file is processed through a reusable pipeline:

1. **Rename** the messy first column (StatCan adds footnote superscripts like `Age group 3 6`) to `Age_Group`
2. **Drop** non-data rows — unit labels (`Persons`, `All ages`) that slip through
3. **Melt** (unpivot) from wide format (years as columns) to long format (one row per year)
4. **Cast** `Population` to numeric — `str.replace(',', '')` strips StatCan comma formatting
5. **Tag** each row with `Gender` (`Male` or `Female`)


In [ ]:
def process_file(file_path, gender_label):
    """
    Full cleaning pipeline for one StatCan population CSV.

    Returns long-format DataFrame:
      Year | Age_Group | Gender | Population
    """
    df = read_statscan_table(file_path)

    # ── Rename messy first column ────────────────────────────
    df.rename(columns={df.columns[0]: "Age_Group"}, inplace=True)

    # ── Drop non-data rows that passed the boundary scan ────
    drop_rows = ["Persons", "All ages", "Median age", "Age group 3 6"]
    df = df[~df["Age_Group"].isin(drop_rows)]
    df = df[df["Age_Group"].notna()]

    # ── Unpivot: wide → long (14 year columns → 14 rows each) ──
    # pd.melt = SQL UNPIVOT
    df_long = df.melt(
        id_vars="Age_Group",
        var_name="Year",
        value_name="Population"
    )

    # ── Cast types ───────────────────────────────────────────
    # Strip comma formatting e.g. "29,115" → 29115
    df_long["Population"] = (
        df_long["Population"]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    df_long["Population"] = pd.to_numeric(df_long["Population"], errors="coerce")
    df_long["Year"]       = pd.to_numeric(df_long["Year"],       errors="coerce")

    # ── Drop any rows that failed to parse ──────────────────
    df_long = df_long.dropna(subset=["Year", "Population"])

    # ── Tag gender ──────────────────────────────────────────
    df_long["Gender"] = gender_label

    return df_long[["Year", "Age_Group", "Gender", "Population"]]


## Step 4 — Run Pipeline: Both Files

Process each gender file and combine into a single unified dataset using `pd.concat` (equivalent to SQL `UNION ALL`).


In [ ]:
men_df   = process_file(men_file,   "Male")
women_df = process_file(women_file, "Female")

final_df = pd.concat([men_df, women_df], ignore_index=True)

# Sort for readability
final_df = final_df.sort_values(by=["Age_Group", "Gender", "Year"])

print(f"Combined shape : {final_df.shape}")
print(f"Year range     : {int(final_df.Year.min())} – {int(final_df.Year.max())}")
print(f"Genders        : {final_df.Gender.unique().tolist()}")
print(f"Age groups     : {final_df.Age_Group.nunique()}")
print()
print(final_df.head())


## Step 5 — Save Clean CSV

The cleaned long-format CSV is the primary output — used by Power BI for the Population Trends page and as input to the forecasting notebook.


In [ ]:
import os
os.makedirs("./clean",    exist_ok=True)
os.makedirs("./database", exist_ok=True)

final_df.to_csv(output_csv, index=False)
print("✅ Clean CSV saved:", output_csv)


## Step 6 — Load to SQLite

SQLite is a file-based database built into Python — no server required. Loading here means you can query the clean data with SQL directly from Python, or connect Power BI via ODBC.

The `DROP TABLE IF EXISTS` before `to_sql()` prevents SQLite locking errors on re-runs.


In [ ]:
conn   = sqlite3.connect(sqlite_db)
cursor = conn.cursor()

# Explicit drop avoids SQLite locking issues on re-run
cursor.execute("DROP TABLE IF EXISTS population;")
conn.commit()

final_df.to_sql("population", conn, index=False)
conn.close()

print("✅ Data loaded into SQLite:", sqlite_db)
print()
print("── Final Output Schema ──────────────────────")
print(final_df.dtypes)
print()
print("── 2025 NS Population Totals ────────────────")
totals = final_df[
    (final_df["Age_Group"] == "0 to 4 years") |
    (final_df["Year"] == 2025)
]
print(
    final_df[final_df["Year"] == 2025]
    .groupby("Gender")["Population"]
    .sum()
    .rename("Total 2025 Population")
)


## ✅ Output Summary

| Output | Location | Rows | Columns |
|--------|----------|------|---------|
| Clean CSV | `./clean/clean_population.csv` | ~588 | 4 |
| SQLite table | `./database/population.db → population` | ~588 | 4 |

**Final schema:** `Year` \| `Age_Group` \| `Gender` \| `Population`

**Feeds into:**
- 📊 Power BI — Population Trends page (area chart, population pyramid)
- 🔮 `Population_Forecast.ipynb` — polynomial regression projection 2026–2035


---

*Nova Scotia Health & Population Analytics · Statistics Canada 17-10-0005-01*
